# Category + subcategory pair prediction
Frozen sentence embeddings + logistic regression. **Grouped evaluation only: five outer test folds and three inner selection folds.** One class represents one valid category/subcategory pair.

Inputs: prepared initial short description + description. Labels: reviewed category and subcategory. No category, priority or escalation outcome is fed as an input. Identical text and incident/problem-linked tickets stay together through connected grouping.

The synthetic dataset contains repeated templates. High scores here demonstrate a working pipeline, not production accuracy or a verified improvement over the current routing system.

In [1]:
from pathlib import Path
import sys, subprocess
import pandas as pd
from IPython.display import display
candidates = [Path.cwd(), Path.cwd() / 'category_pair_model', Path.cwd() / 'ticket_data_preparation/category_pair_model']
MODEL_DIR = next((p for p in candidates if (p / 'train_category_pair.py').exists()), None)
if MODEL_DIR is None:
    raise FileNotFoundError('Open the notebook from the extracted ticket_data_preparation folder.')
print('Model folder:', MODEL_DIR)


Model folder: /workspace/scratch/ec4ca456f2d2/project/ticket_data_preparation/category_pair_model


## Train and evaluate
Run the shared training script. C is selected by inner out-of-fold pair macro F1; each outer test fold is evaluated once. Scores choose the highest-probability pair; the escalation recall threshold does not apply. The final deployable model is fitted separately after evaluation.

Rare labels may have fewer examples than the fold count. Their held-out predictions are retained; see pair support and unseen-pair counts. Stratification is approximate with intact groups.

In [2]:
completed = subprocess.run([sys.executable, str(MODEL_DIR / 'train_category_pair.py')], capture_output=True, text=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
completed.check_returncode()


2625 tickets; 39 pairs; 803 connected groups
Fold 1: C=10.0, pair accuracy=0.9962
Fold 2: C=10.0, pair accuracy=1.0000
Fold 3: C=100.0, pair accuracy=1.0000
Fold 4: C=100.0, pair accuracy=1.0000
Fold 5: C=100.0, pair accuracy=1.0000
                 model  tickets  pair_accuracy  pair_macro_f1  pair_weighted_f1  category_accuracy  category_macro_f1  category_weighted_f1  subcategory_accuracy  subcategory_macro_f1  subcategory_weighted_f1
     embedding_lr_pair     2625         0.9992         0.9995            0.9992              1.000             1.0000                 1.000                0.9992                0.9995                   0.9992
training_majority_pair     2625         0.1032         0.0048            0.0193              0.269             0.0424                 0.114                0.1032                0.0048                   0.0193


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5113.43it/s]
/opt/codex/runtimes/codex-primary-runtime/dependencies/python/lib/pyt

## Held-out grouped results
Exact-pair accuracy requires both outputs to match. Macro F1 gives equal weight to evaluated classes. The majority baseline picks the most common training pair separately in each outer fold.

In [3]:
OUT = MODEL_DIR / 'results'
display(pd.read_csv(OUT / 'grouped_metrics.csv'))
display(pd.read_csv(OUT / 'grouped_fold_metrics.csv'))
display(pd.read_csv(OUT / 'misclassified_pairs.csv')[['ticket_id', 'target_category', 'target_subcategory', 'predicted_category', 'predicted_subcategory']])


model,tickets,pair_accuracy,pair_macro_f1,pair_weighted_f1,category_accuracy,category_macro_f1,category_weighted_f1,subcategory_accuracy,subcategory_macro_f1,subcategory_weighted_f1
embedding_lr_pair,2625,0.999238,0.999542,0.999238,1.000000,1.00000,1.000000,0.999238,0.999542,0.999238
training_majority_pair,2625,0.103238,0.004799,0.019321,0.268952,0.04239,0.114008,0.103238,0.004799,0.019321


fold,selected_C,train_tickets,test_tickets,pair_accuracy,pair_macro_f1,unseen_pair_tickets
1,10.0,2096,529,0.996219,0.997653,0
2,10.0,2087,538,1.000000,1.000000,0
3,100.0,2089,536,1.000000,1.000000,0
4,100.0,2118,507,1.000000,1.000000,0
5,100.0,2110,515,1.000000,1.000000,0


ticket_id,target_category,target_subcategory,predicted_category,predicted_subcategory
TKT0100292,Investor Portal,Document not visible,Investor Portal,Navigation question
TKT0100889,Investor Portal,Document not visible,Investor Portal,Navigation question


## Predict new tickets
Uses the final classifier trained on all eligible rows. These example predictions are demonstrations, not additional test results. Pair confidence is uncalibrated. Use the same initial-text cleaning as training before prediction.

In [4]:
prediction = subprocess.run([sys.executable, str(MODEL_DIR / 'predict_category_pair.py')], capture_output=True, text=True)
print(prediction.stdout)
if prediction.returncode:
    print(prediction.stderr)
prediction.check_returncode()


ticket_id               short_description                                                              description predicted_category     predicted_subcategory  pair_confidence
   NEW001 Instrument data looks incorrect The coupon and maturity for an instrument look incorrect. Please verify.          Analytics  Instrument data question         0.995118
   NEW002       Model assignment question               How can we assign one model to a group of client accounts?             Wealth Model assignment question         0.996970



## Verify grouping and outputs
Tests check intact groups, one held-out prediction per eligible ticket, no train/test group overlap, valid predicted pairs, and final model label coverage. Existing escalation notebooks are separate and unchanged.

In [5]:
checks = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(MODEL_DIR), '-p', 'test_category_pair.py', '-v'], capture_output=True, text=True)
print(checks.stdout + checks.stderr)
checks.check_returncode()


test_group_integrity (test_category_pair.PairTests.test_group_integrity) ... ok
test_held_out_predictions_and_saved_model (test_category_pair.PairTests.test_held_out_predictions_and_saved_model) ... ok
test_pair_roundtrip (test_category_pair.PairTests.test_pair_roundtrip) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.108s

OK



# Additional experiment: semantic template-family grouping
The original experiment above is retained. This extension changes group IDs, while retaining five outer and three inner grouped folds. Frozen embeddings link descriptions at cosine similarity >= 0.90, then merge these links with existing exact-text and incident/problem groups.

**0.90 is an exploratory starting threshold, not a validated template rule.** Review wording in `threshold_review_examples.csv`; do not tune the threshold to obtain a desired accuracy. Groups are built without category labels. Connected components are transitive and can become large.

**Observed limitation:** the rule creates a 2,063-ticket component (78.59% of this dataset). The largest test fold contains 2,088 tickets with only 537 training tickets, and 254 unseen-pair test tickets. Treat this as an imbalanced stress test requiring group review, not a clean replacement accuracy estimate.

In [6]:
template_run = subprocess.run([sys.executable, str(MODEL_DIR / 'train_template_family.py'), '--threshold', '0.90'], capture_output=True, text=True)
print(template_run.stdout)
if template_run.stderr:
    print(template_run.stderr)
template_run.check_returncode()


2625 tickets; 39 pairs; 103 connected groups
Fold 1: C=100.0, pair accuracy=0.6651
Fold 2: C=100.0, pair accuracy=1.0000
Fold 3: C=100.0, pair accuracy=1.0000
Fold 4: C=100.0, pair accuracy=0.7538
Fold 5: C=100.0, pair accuracy=1.0000
                 model  tickets  pair_accuracy  pair_macro_f1  pair_weighted_f1  category_accuracy  category_macro_f1  category_weighted_f1  subcategory_accuracy  subcategory_macro_f1  subcategory_weighted_f1
     embedding_lr_pair     2625         0.7771         0.7315            0.7674             0.8011             0.7781                0.8081                0.7771                0.7315                   0.7674
training_majority_pair     2625         0.0206         0.0034            0.0138             0.0461             0.0188                0.0492                0.0206                0.0034                   0.0138
           experiment                  model  tickets  pair_accuracy  pair_macro_f1  pair_weighted_f1  category_accuracy  category_macro_f

In [7]:
import json
TEMPLATE_OUT = MODEL_DIR / 'results_template_family'
print(json.dumps(json.loads((TEMPLATE_OUT / 'grouping_diagnostics.json').read_text()), indent=2))
display(pd.read_csv(TEMPLATE_OUT / 'comparison_with_original.csv'))
display(pd.read_csv(TEMPLATE_OUT / 'grouped_fold_metrics.csv'))
display(pd.read_csv(TEMPLATE_OUT / 'template_group_sizes.csv').sort_values('tickets', ascending=False).head(10))
display(pd.read_csv(TEMPLATE_OUT / 'threshold_review_examples.csv').head(10))


{
  "threshold": 0.9,
  "original_groups": 803,
  "new_groups": 103,
  "unique_texts": 1046,
  "similarity_edges": 6421,
  "largest_group_tickets": 2063,
  "largest_group_fraction": 0.7859047619047619,
  "rule": "cosine >= threshold, union with original groups, transitive connected components",
  "status": "Exploratory semantic proxy, not verified source template IDs. Threshold fixed before evaluation, not selected for accuracy."
}


experiment,model,tickets,pair_accuracy,pair_macro_f1,pair_weighted_f1,category_accuracy,category_macro_f1,category_weighted_f1,subcategory_accuracy,subcategory_macro_f1,subcategory_weighted_f1
original_grouping,embedding_lr_pair,2625,0.999238,0.999542,0.999238,1.000000,1.000000,1.000000,0.999238,0.999542,0.999238
original_grouping,training_majority_pair,2625,0.103238,0.004799,0.019321,0.268952,0.042390,0.114008,0.103238,0.004799,0.019321
template_family_proxy,embedding_lr_pair,2625,0.777143,0.731476,0.767359,0.801143,0.778114,0.808125,0.777143,0.731476,0.767359
template_family_proxy,training_majority_pair,2625,0.020571,0.003427,0.013799,0.046095,0.018794,0.049175,0.020571,0.003427,0.013799


fold,selected_C,train_tickets,test_tickets,pair_accuracy,pair_macro_f1,unseen_pair_tickets
1,100.0,2413,212,0.665094,0.638854,0
2,100.0,2516,109,1.000000,1.000000,0
3,100.0,2517,108,1.000000,1.000000,0
4,100.0,537,2088,0.753831,0.673815,254
5,100.0,2517,108,1.000000,1.000000,0


evaluation_group,tickets,unique_texts,original_groups
TF002490,2063,769,556
TF002040,189,95,65
TF001550,34,8,8
TF002318,26,10,10
TF002459,22,5,5
TF002084,19,6,6
TF001921,15,4,4
TF001463,15,4,4
TF002330,13,4,4
TF001399,9,4,4


cosine_similarity,merged_directly,text_a,text_b
0.900008,True,NAV difference for Fund FUND The Q2 2024 NAV for Fund FUND differs from the manager report by 17.6m. Please investigate.,"Q2 2024 NAV does not match manager report Hello, The Q2 2024 NAV for Fund FUND differs from the manager report by 11.4m. Please investigate."
0.899976,False,The Q2 2024 NAV for Fund FUND differs from the manager report by 2.7m. Please investigate.,NAV difference for Fund FUND The Q4 2023 NAV for Fund FUND differs from the manager report by 17.7m. Please investigate.
0.899969,False,Capital call notice question - Fund FUND The capital call notice for Fund FUND shows 10.8m; we expected a different amount. Please confirm.,"Capital call amount query Hi, The capital call notice for Fund FUND shows 13.6m; we expected a different amount. Please confirm."
0.900031,True,"NAV difference for Fund FUND Hello, The Q4 2023 NAV for Fund FUND differs from the manager report by 1.6m. Please investigate.","Q1 2024 NAV does not match manager report Hi, The Q1 2024 NAV for Fund FUND differs from the manager report by 17.1m. Please investigate."
0.900040,True,Q1 2024 NAV does not match manager report The Q1 2024 NAV for Fund FUND differs from the manager report by 19.5m. Please investigate.,"NAV difference for Fund FUND Good morning, The Q2 2024 NAV for Fund FUND differs from the manager report by 7.4m. Please investigate."
0.900043,True,Approval workflow update request Please add a second approval step for transactions above 9.3m.,Approval workflow update request Please add a second approval step for transactions above 19.3m.
0.899947,False,"Coverage request - Fund FUND Hi, Please add Fund FUND to our data coverage starting Q4 2023.","Hello, Please add Fund FUND to our data coverage starting Q2 2024."
0.899935,False,"Hello, The Q4 2023 NAV for Fund FUND differs from the manager report by 9.2m. Please investigate.","NAV difference for Fund FUND Good morning, The Q2 2024 NAV for Fund FUND differs from the manager report by 18.7m. Please investigate."
0.899931,False,"Q1 2024 NAV does not match manager report Good morning, The Q1 2024 NAV for Fund FUND differs from the manager report by 11.0m. Please investigate.",NAV difference for Fund FUND The Q1 2024 NAV for Fund FUND differs from the manager report by 9.0m. Please investigate.
0.899919,False,"Capital call notice question - Fund FUND Hello, The capital call notice for Fund FUND shows 8.4m; we expected a different amount. Please confirm.",Capital call amount query The capital call notice for Fund FUND shows 2.3m; we expected a different amount. Please confirm.


The existing prediction command still defaults to the original model. To use the added final model, pass `--model category_pair_model/results_template_family/category_pair_model.joblib` from the project root. Final models are fitted on all eligible rows after grouped selection; their training accuracy is not used for either evaluation.

Both experiments use synthetic data. Similarity links are proxies for source templates and may connect different issues. Inspect label coverage, pair support and group sizes alongside the score.

In [8]:
checks = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(MODEL_DIR), '-p', 'test_*.py', '-v'], capture_output=True, text=True)
print(checks.stdout + checks.stderr)
checks.check_returncode()


test_group_integrity (test_category_pair.PairTests.test_group_integrity) ... ok
test_held_out_predictions_and_saved_model (test_category_pair.PairTests.test_held_out_predictions_and_saved_model) ... ok
test_pair_roundtrip (test_category_pair.PairTests.test_pair_roundtrip) ... ok
test_no_label_dependency (test_template_groups.TemplateTests.test_no_label_dependency) ... ok
test_similarity_and_original_links_are_both_preserved (test_template_groups.TemplateTests.test_similarity_and_original_links_are_both_preserved) ... ok
test_template_oof_integrity (test_template_groups.TemplateTests.test_template_oof_integrity) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.138s

OK



# Third experiment: template-only grouping
Uses exact text and cosine similarity >= 0.90 only. Incident/problem links do not constrain folds. Same eligible tickets, frozen MiniLM embeddings, C grid, five outer and three inner grouped folds as before. Template families are approximate connected components, so chaining still applies. This does NOT implement complete-linkage/all-pairs clustering.

Groups are built without labels. The threshold remains fixed; no threshold search using model accuracy. Related incidents may appear across folds: this experiment assesses held-out wording families, not unseen events. Results remain exploratory on synthetic data.

In [9]:
only_run = subprocess.run([sys.executable, str(MODEL_DIR / 'train_template_only.py')], capture_output=True, text=True)
print(only_run.stdout)
if only_run.stderr:
    print(only_run.stderr)
only_run.check_returncode()


2625 tickets; 39 pairs; 195 connected groups
Fold 1: C=100.0, pair accuracy=0.9851
Fold 2: C=100.0, pair accuracy=0.9080
Fold 3: C=100.0, pair accuracy=1.0000
Fold 4: C=100.0, pair accuracy=0.8640
Fold 5: C=100.0, pair accuracy=0.9100
                 model  tickets  pair_accuracy  pair_macro_f1  pair_weighted_f1  category_accuracy  category_macro_f1  category_weighted_f1  subcategory_accuracy  subcategory_macro_f1  subcategory_weighted_f1
     embedding_lr_pair     2625         0.9337         0.8769            0.9234             0.9684             0.9369                0.9663                0.9337                0.8769                   0.9234
training_majority_pair     2625         0.1032         0.0048            0.0193             0.2690             0.0424                0.1140                0.1032                0.0048                   0.0193
               experiment                  model  tickets  pair_accuracy  pair_macro_f1  pair_weighted_f1  category_accuracy  category_mac

In [10]:
import json
ONLY_OUT = MODEL_DIR / 'results_template_only'
print(json.dumps(json.loads((ONLY_OUT / 'grouping_diagnostics.json').read_text()), indent=2))
display(pd.read_csv(ONLY_OUT / 'comparison_all_groupings.csv'))
display(pd.read_csv(ONLY_OUT / 'grouped_fold_metrics.csv'))
display(pd.read_csv(ONLY_OUT / 'template_group_sizes.csv').sort_values('tickets', ascending=False).head(10))
event_audit = pd.read_csv(ONLY_OUT / 'event_cross_fold_audit.csv')
print('Event groups crossing outer test folds:', int((event_audit.outer_test_folds > 1).sum()))
display(pd.read_csv(ONLY_OUT / 'misclassified_pairs.csv')[['ticket_id', 'target_category', 'target_subcategory', 'predicted_category', 'predicted_subcategory']].head(10))


{
  "threshold": 0.9,
  "original_groups": 803,
  "new_groups": 195,
  "unique_texts": 1046,
  "similarity_edges": 6421,
  "largest_group_tickets": 126,
  "largest_group_fraction": 0.048,
  "preserve_event_links": false,
  "rule": "cosine >= threshold plus exact text only; transitive connected components",
  "status": "Exploratory semantic proxy, not verified source template IDs. Threshold fixed before evaluation, not selected for accuracy."
}
Event groups crossing outer test folds: 101


experiment,model,tickets,pair_accuracy,pair_macro_f1,pair_weighted_f1,category_accuracy,category_macro_f1,category_weighted_f1,subcategory_accuracy,subcategory_macro_f1,subcategory_weighted_f1
original_grouping,embedding_lr_pair,2625,0.999238,0.999542,0.999238,1.000000,1.000000,1.000000,0.999238,0.999542,0.999238
original_grouping,training_majority_pair,2625,0.103238,0.004799,0.019321,0.268952,0.042390,0.114008,0.103238,0.004799,0.019321
combined_similarity_event,embedding_lr_pair,2625,0.777143,0.731476,0.767359,0.801143,0.778114,0.808125,0.777143,0.731476,0.767359
combined_similarity_event,training_majority_pair,2625,0.020571,0.003427,0.013799,0.046095,0.018794,0.049175,0.020571,0.003427,0.013799
template_only,embedding_lr_pair,2625,0.933714,0.876855,0.923431,0.968381,0.936876,0.966347,0.933714,0.876855,0.923431
template_only,training_majority_pair,2625,0.103238,0.004799,0.019321,0.268952,0.042390,0.114008,0.103238,0.004799,0.019321


fold,selected_C,train_tickets,test_tickets,pair_accuracy,pair_macro_f1,unseen_pair_tickets
1,100.0,2088,537,0.985102,0.990821,0
2,100.0,2103,522,0.908046,0.910719,14
3,100.0,2103,522,1.000000,1.000000,0
4,100.0,2103,522,0.863985,0.856482,0
5,100.0,2103,522,0.909962,0.854515,0


evaluation_group,tickets,unique_texts,original_groups
TF002588,126,104,86
TF002061,87,16,16
TF000548,84,23,23
TF001805,81,17,16
TF000464,76,18,17
TF001039,63,14,13
TF002427,59,57,51
TF001729,58,10,7
TF002431,57,23,23
TF001597,54,13,13


ticket_id,target_category,target_subcategory,predicted_category,predicted_subcategory
TKT0100036,Workflow & Setup,Approval workflow change,Workflow & Setup,Stuck approval
TKT0100039,Investor Documents,Capital call notice,Investor Documents,Distribution notice
TKT0100044,Investor Documents,Capital call notice,Investor Documents,Distribution notice
TKT0100058,Investor Documents,Capital call notice,Investor Documents,Distribution notice
TKT0100082,Investor Documents,Capital call notice,Investor Documents,Distribution notice
TKT0100111,Investor Portal,Notification emails,Investor Portal,Document not visible
TKT0100168,Fund Accounting,Duplicate transaction,Investor Documents,Capital call notice
TKT0100187,Wealth,Order not progressing,Workflow & Setup,Stuck approval
TKT0100189,Investor Documents,Capital call notice,Fund Accounting,Duplicate transaction
TKT0100208,Investor Portal,Notification emails,Investor Portal,Document not visible


In [11]:
checks = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(MODEL_DIR), '-p', 'test_*.py', '-v'], capture_output=True, text=True)
print(checks.stdout + checks.stderr)
checks.check_returncode()


test_group_integrity (test_category_pair.PairTests.test_group_integrity) ... ok
test_held_out_predictions_and_saved_model (test_category_pair.PairTests.test_held_out_predictions_and_saved_model) ... ok
test_pair_roundtrip (test_category_pair.PairTests.test_pair_roundtrip) ... ok
test_no_label_dependency (test_template_groups.TemplateTests.test_no_label_dependency) ... ok
test_similarity_and_original_links_are_both_preserved (test_template_groups.TemplateTests.test_similarity_and_original_links_are_both_preserved) ... ok
test_template_oof_integrity (test_template_groups.TemplateTests.test_template_oof_integrity) ... ok
test_event_links_ignored_but_duplicates_preserved (test_template_only.TemplateOnlyTests.test_event_links_ignored_but_duplicates_preserved) ... ok
test_held_out_groups_and_similarity_links (test_template_only.TemplateOnlyTests.test_held_out_groups_and_similarity_links) ... ok

----------------------------------------------------------------------
Ran 8 tests in 0.370s

OK


The original and combined experiments above remain available. To predict with the template-only final model, use `predict_category_pair.py --model category_pair_model/results_template_only/category_pair_model.joblib` from the project root, prefixed by `python category_pair_model/` for the script path. The final model is fitted to all eligible rows after grouped selection; evaluation uses only outer held-out predictions.